# Verificação e avaliação da Secretária IA
Este notebook usa os mesmos módulos do terminal e do Streamlit. As respostas abaixo são **fixtures manuais**, sem chamada de LLM ou LangSmith.

In [ ]:
from pathlib import Path
import sys
raiz = Path.cwd()
if not (raiz / 'src' / 'secretaria').exists():
    raiz = raiz.parent
assert (raiz / 'src' / 'secretaria').exists(), 'Execute na pasta do projeto ou em notebooks/'
sys.path.insert(0, str(raiz / 'src'))
from secretaria.avaliacao import carregar_casos, avaliar_caso
from secretaria.verificador import verificar_deterministico
casos = carregar_casos(raiz)
len(casos)

In [ ]:
resultados = [avaliar_caso(c, c['fixture_resposta'], c['fixture_ferramentas']) for c in casos]
for r in resultados:
    print(r['id'], 'ferramentas=', r['nota_ferramentas'], 'resposta_fixture=', r['nota_resposta'])
assert next(r for r in resultados if r['id'] == 'oferta_consulta')['nota_resposta'] == 0

A igualdade das listas de ferramentas é calculada em Python. A nota de resposta local vem do gabarito da fixture, não de um juiz executado. Na execução com API, um juiz semântico usa critérios explícitos e a revisão humana continua necessária.

In [ ]:
import json
from types import SimpleNamespace
retorno = SimpleNamespace(type='tool', name='consultar_mensagens_estruturado', content=json.dumps({'status':'ok','plano':{'tipo':'contagem'},'total':1}))
print(verificar_deterministico('Quantas mensagens?', 'Foram encontradas 3 mensagens.', [retorno], 'consulta'))
print(verificar_deterministico('Quantas mensagens?', 'Foi encontrada 1 mensagem.', [retorno], 'consulta'))